In [ ]:
!apt update -q && apt-get install -qy openjdk-21-jdk-headless > /dev/null 2>&1

import subprocess
from IPython.core.magic import register_cell_magic

class PersistentJShell:
    def __init__(self):
        self.proc = subprocess.Popen(
            ["jshell", "-s"],
            stdin=subprocess.PIPE,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1
        )

    def execute(self, code):
        if self.proc.poll() is not None:
            raise RuntimeError("JShell process has terminated.")

        marker = "___JSHELL_CELL_END___"
        payload = code.strip() + f'\nSystem.out.println("{marker}");\n'
        self.proc.stdin.write(payload)
        self.proc.stdin.flush()

        lines = []
        while True:
            line = self.proc.stdout.readline()
            if not line:
                break
            if marker in line:
                break
            lines.append(line)

        return "".join(lines)

_jshell_instance = PersistentJShell()

@register_cell_magic
def jshell(line, cell):
    out = _jshell_instance.execute(cell)
    if out:
        print(out, end="")


In [ ]:
%%jshell

void runPy(String code) throws Exception {
    String marker = "___END_OF_PYTHON_CELL___";

    // Base64 encode code payload to bypass all quote & newline syntax errors
    String b64 = java.util.Base64.getEncoder().encodeToString(code.getBytes(java.nio.charset.StandardCharsets.UTF_8));

    // Guarantee marker printing using try...finally and display trailing expressions using ast
    String payload = String.format(
        "import base64, ast, traceback\n" +
        "try:\n" +
        "    code_str = base64.b64decode('%s').decode('utf-8')\n" +
        "    tree = ast.parse(code_str)\n" +
        "    if tree.body and isinstance(tree.body[-1], ast.Expr):\n" +
        "        exec(compile(ast.Module(body=tree.body[:-1], type_ignores=[]), '<string>', 'exec'), globals())\n" +
        "        res = eval(compile(ast.Expression(body=tree.body[-1].value), '<string>', 'eval'), globals())\n" +
        "        if res is not None:\n" +
        "            print(res)\n" +
        "    else:\n" +
        "        exec(compile(tree, '<string>', 'exec'), globals())\n" +
        "except Exception:\n" +
        "    traceback.print_exc()\n" +
        "finally:\n" +
        "    print('%s', flush=True)\n",
        b64, marker
    );

    pyIn.println(payload);

    String line;
    while ((line = pyOut.readLine()) != null) {
        if (line.contains(marker)) break;
        System.out.println(line);
    }
}

import java.io.BufferedReader;
import java.io.InputStreamReader;
import java.io.PrintStream;
import java.io.IOException;

Process pythonProcess;
PrintStream pyIn;
BufferedReader pyOut;

try {
    ProcessBuilder pb = new ProcessBuilder("python3", "-i", "-u");
    // Redirect stderr to stdout to prevent pipe deadlock and capture Python errors
    pb.redirectErrorStream(true);
    pythonProcess = pb.start();

    pyIn = new PrintStream(pythonProcess.getOutputStream(), true);
    pyOut = new BufferedReader(new InputStreamReader(pythonProcess.getInputStream()));

    System.out.println("Python interpreter initialized safely.");
} catch (IOException e) {
    e.printStackTrace();
}

In [ ]:
%%jshell

runPy("""
# Google Colab setup: fetch this repository and use this notebook's directory.
from pathlib import Path
import os

REPO_ROOT = Path('/content/BITS_programming')
if not REPO_ROOT.exists():os.system('git clone https://github.com/aqwertyuiop48/BITS_programming.git /content/BITS_programming');

NOTEBOOK_DIR = REPO_ROOT / 'assignments/assignment_13/class1/class1'
os.chdir(NOTEBOOK_DIR)
print(f'Working directory: {NOTEBOOK_DIR}')
""");


# Part 3: Adversarial Robustness Testing

## Overview
In this notebook, we will:
1. Test model robustness against Gaussian noise perturbations
2. Apply adversarial attacks using FGSM (Fast Gradient Sign Method)
3. Visualize adversarial examples
4. Measure robustness metrics
5. Understand model vulnerabilities

## Key Concepts
- **Adversarial Examples**: Inputs designed to fool the model
- **Perturbation**: Small, imperceptible changes that cause misclassification
- **Robustness Score**: Accuracy under adversarial conditions
- **FGSM Attack**: Uses gradient information to craft adversarial examples

## What is Adversarial Robustness?

**Adversarial robustness** measures how well a model performs when faced with adversarial perturbations—small, intentional changes to inputs designed to deceive the model.

### Why is it Important?
- Safety-critical systems (autonomous driving, medical imaging) need robust models
- Small sensor noise or image compression can cause failures
- Malicious actors may craft adversarial inputs

### Types of Perturbations:
1. **Gaussian Noise**: Random noise added to inputs
2. **FGSM Attack**: Gradient-based adversarial attack
3. **Pixel-level Perturbations**: Strategic small changes

### Robustness Metrics:
- Clean Accuracy: Accuracy on unperturbed images
- Noisy Accuracy: Accuracy on noise-perturbed images
- Robust Accuracy: Accuracy on adversarial examples
- Robustness Score: (Robust Acc / Clean Acc) × 100%

In [ ]:
%%jshell

runPy("""
import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# PyTorch imports
import torch
import torch.nn as nn
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
from torchvision.datasets import ImageFolder
from torchvision.models import resnet18

from tqdm import tqdm

# Set random seeds
np.random.seed(42)
torch.manual_seed(42)

print("✅ All libraries imported successfully")
""");


## Step 1: Setup & Load Pre-trained Model

In [ ]:
%%jshell

runPy("""
# Device configuration
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Dataset paths
DATASET_PATH = str(REPO_ROOT / "module_4/dataset/dataset")
TEST_PATH = os.path.join(DATASET_PATH, "test")

# Data transformations
transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

# Load test dataset
print("Loading test dataset...")
test_dataset = ImageFolder(TEST_PATH, transform=transform)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=True, num_workers=0)

class_names = test_dataset.classes
num_classes = len(class_names)

print(f"✅ Classes: {', '.join(class_names)}")
print(f"✅ Test samples: {len(test_dataset)}")
""");


## Step 2: Define Perturbation Functions

In [ ]:
%%jshell

runPy("""
def add_gaussian_noise(images, noise_std=0.1):
    """
    Add Gaussian noise to images.

    Args:
        images: Tensor of shape (B, C, H, W)
        noise_std: Standard deviation of Gaussian noise

    Returns:
        Noisy images (clipped to valid range)
    """
    noise = torch.randn_like(images) * noise_std
    noisy_images = images + noise
    # Clip to [-1, 1] for normalized images (ImageNet normalization)
    return torch.clamp(noisy_images, -1, 1)

def fgsm_attack(model, images, labels, device, epsilon=0.05):
    """
    Fast Gradient Sign Method (FGSM) attack.

    The attack computes the gradient of the loss with respect to the input
    and moves in the direction that increases the loss (to fool the model).

    Args:
        model: Neural network model
        images: Input images
        labels: True labels
        device: Device to run on
        epsilon: Attack strength (max perturbation per pixel)

    Returns:
        Adversarial images
    """
    images.requires_grad = True

    # Forward pass
    outputs = model(images)
    loss = nn.CrossEntropyLoss()(outputs, labels)

    # Backward pass - compute gradients
    model.zero_grad()
    loss.backward()

    # Get gradient with respect to input
    data_grad = images.grad.data

    # Create adversarial images by following gradient direction
    sign_data_grad = data_grad.sign()
    perturbed_images = images + epsilon * sign_data_grad

    # Clip to valid range
    perturbed_images = torch.clamp(perturbed_images, -1, 1)

    return perturbed_images.detach()

print("✅ Perturbation functions defined")
""");


## Step 3: Create & Load Pre-trained Model

We'll use the model trained in Part 1. You should have trained a model and saved it, or we'll train a fresh one here.

In [ ]:
%%jshell

runPy("""
# Create model
model = resnet18(pretrained=True)
model.fc = nn.Linear(model.fc.in_features, num_classes)
model = model.to(device)

# For demonstration, we'll use the model with weights from training
# In practice, you would load a pre-trained checkpoint
model.eval()

print("✅ Model loaded")
""");


## Step 4: Evaluate Robustness Against Gaussian Noise

In [ ]:
%%jshell

runPy("""
def evaluate_robustness(model, loader, device, perturbation_func, perturbation_args=None):
    """
    Evaluate model on perturbed images.

    Args:
        model: Neural network model
        loader: Data loader
        device: Device to run on
        perturbation_func: Function to apply perturbations
        perturbation_args: Arguments for perturbation function

    Returns:
        Accuracy on perturbed images
    """
    if perturbation_args is None:
        perturbation_args = {}

    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in tqdm(loader, desc="Evaluating robustness", leave=False):
            images, labels = images.to(device), labels.to(device)

            # Apply perturbation
            if perturbation_func == "gaussian":
                perturbed = add_gaussian_noise(images, **perturbation_args)
            else:
                perturbed = images

            # Evaluate on perturbed images
            outputs = model(perturbed)
            _, predicted = torch.max(outputs, 1)

            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    accuracy = 100 * correct / total
    return accuracy

# Test with different noise levels
print("\n" + "="*70)
print("🔊 GAUSSIAN NOISE ROBUSTNESS TEST")
print("="*70)

noise_levels = [0.0, 0.05, 0.1, 0.15, 0.2, 0.3]
accuracies_noise = []

for noise_std in noise_levels:
    if noise_std == 0:
        # Clean accuracy
        acc = evaluate_robustness(model, test_loader, device, "none")
        print(f"Clean (no noise): {acc:.2f}%")
    else:
        acc = evaluate_robustness(model, test_loader, device, "gaussian", {'noise_std': noise_std})
        print(f"Noise Std {noise_std}: {acc:.2f}%")

    accuracies_noise.append(acc)

print("="*70)
""");


## Step 5: Test Against FGSM Attacks

In [ ]:
%%jshell

runPy("""
def evaluate_adversarial_robustness(model, loader, device, epsilon_values):
    """
    Evaluate model robustness against FGSM attacks with different epsilon values.

    Args:
        model: Neural network model
        loader: Data loader
        device: Device to run on
        epsilon_values: List of epsilon values to test

    Returns:
        Dataset of accuracies on adversarial examples
    """
    accuracies = []

    model.eval()

    for epsilon in epsilon_values:
        correct = 0
        total = 0

        for images, labels in tqdm(loader, desc=f"Testing epsilon={epsilon:.3f}", leave=False):
            images, labels = images.to(device), labels.to(device)

            # Generate adversarial examples
            adv_images = fgsm_attack(model, images.clone(), labels, device, epsilon=epsilon)

            # Evaluate on adversarial images
            outputs = model(adv_images)
            _, predicted = torch.max(outputs, 1)

            total += labels.size(0)
            correct += (predicted == labels).sum().item()

        accuracy = 100 * correct / total
        accuracies.append(accuracy)

    return accuracies

# Test with different epsilon values
print("\n" + "="*70)
print("⚔️  FGSM ADVERSARIAL ATTACK ROBUSTNESS TEST")
print("="*70)

epsilon_values = [0.0, 0.01, 0.03, 0.05, 0.1, 0.15]
accuracies_fgsm = evaluate_adversarial_robustness(model, test_loader, device, epsilon_values)

for eps, acc in zip(epsilon_values, accuracies_fgsm):
    if eps == 0:
        print(f"Clean (no attack):     {acc:.2f}%")
    else:
        print(f"FGSM ε={eps:.4f}:        {acc:.2f}%")

print("="*70)
""");


## Step 6: Visualize Perturbations & Adversarial Examples

In [ ]:
%%jshell

runPy("""
# Get sample images from data loader
sample_images, sample_labels = next(iter(test_loader))
sample_images = sample_images[:8].to(device)  # Take first 8 images
sample_labels = sample_labels[:8].to(device)

# De-normalization function
def denormalize(images):
    """Convert normalized images back to [0, 1] range for visualization"""
    images = images.clone()
    images[:, 0] = images[:, 0] * 0.229 + 0.485
    images[:, 1] = images[:, 1] * 0.224 + 0.456
    images[:, 2] = images[:, 2] * 0.225 + 0.406
    return torch.clamp(images, 0, 1)

# Create visualization
fig, axes = plt.subplots(3, 8, figsize=(16, 6))
fig.suptitle('Adversarial Perturbation Visualization', fontsize=16, fontweight='bold')

# Row 1: Clean images
clean_images = denormalize(sample_images.cpu())
for idx, ax in enumerate(axes[0]):
    img_array = clean_images[idx].permute(1, 2, 0).numpy()
    ax.imshow(img_array)
    ax.set_title(f"Clean", fontsize=10)
    ax.axis('off')

# Row 2: Gaussian noise perturbations
noisy_images = add_gaussian_noise(sample_images, noise_std=0.15)
noisy_images_viz = denormalize(noisy_images.cpu())
for idx, ax in enumerate(axes[1]):
    img_array = noisy_images_viz[idx].permute(1, 2, 0).numpy()
    ax.imshow(img_array)
    ax.set_title(f"Gaussian Noise", fontsize=10)
    ax.axis('off')

# Row 3: FGSM adversarial examples
adv_images = fgsm_attack(model, sample_images.clone(), sample_labels, device, epsilon=0.05)
adv_images_viz = denormalize(adv_images.cpu())
for idx, ax in enumerate(axes[2]):
    img_array = adv_images_viz[idx].permute(1, 2, 0).numpy()
    ax.imshow(img_array)
    ax.set_title(f"FGSM (ε=0.05)", fontsize=10)
    ax.axis('off')

plt.tight_layout()
plt.savefig('adversarial_examples.png', dpi=150, bbox_inches='tight')
plt.savefig('/content/regression_plot.png')

print("✅ Adversarial examples visualization saved")
""");


## Step 7: Plot Robustness Curves

In [ ]:
%%jshell

runPy("""
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Model Robustness Analysis', fontsize=16, fontweight='bold')

# Plot 1: Gaussian Noise Robustness
ax = axes[0]
ax.plot(noise_levels, accuracies_noise, marker='o', linewidth=3, markersize=8, color='#FF6B6B')
ax.set_xlabel('Noise Std Dev', fontsize=12, fontweight='bold')
ax.set_ylabel('Accuracy (%)', fontsize=12, fontweight='bold')
ax.set_title('Robustness to Gaussian Noise', fontsize=13, fontweight='bold')
ax.grid(True, alpha=0.3)
ax.set_ylim([0, 105])

# Add annotations
for x, y in zip(noise_levels, accuracies_noise):
    ax.annotate(f'{y:.1f}%', (x, y), textcoords="offset points", xytext=(0,10), ha='center', fontsize=9)

# Plot 2: FGSM Attack Robustness
ax = axes[1]
ax.plot(epsilon_values, accuracies_fgsm, marker='s', linewidth=3, markersize=8, color='#4ECDC4')
ax.set_xlabel('Attack Strength (ε)', fontsize=12, fontweight='bold')
ax.set_ylabel('Accuracy (%)', fontsize=12, fontweight='bold')
ax.set_title('Robustness to FGSM Attacks', fontsize=13, fontweight='bold')
ax.grid(True, alpha=0.3)
ax.set_ylim([0, 105])

# Add annotations
for x, y in zip(epsilon_values, accuracies_fgsm):
    ax.annotate(f'{y:.1f}%', (x, y), textcoords="offset points", xytext=(0,10), ha='center', fontsize=9)

plt.tight_layout()
plt.savefig('robustness_curves.png', dpi=150, bbox_inches='tight')
plt.savefig('/content/regression_plot.png')

print("✅ Robustness curves saved")
""");


## Step 8: Calculate Robustness Metrics

In [ ]:
%%jshell

runPy("""
# Extract clean accuracy
clean_acc_noise = accuracies_noise[0]  # First entry is clean accuracy
clean_acc_fgsm = accuracies_fgsm[0]

# Calculate robustness scores
print("\n" + "="*70)
print("📊 ADVERSARIAL ROBUSTNESS METRICS")
print("="*70)

print(f"\n✅ Baseline (Clean) Accuracy: {clean_acc_noise:.2f}%")

print("\n🔊 GAUSSIAN NOISE ROBUSTNESS:")
print(f"  Noise Std=0.05  → Accuracy: {accuracies_noise[1]:.2f}% | Drop: {clean_acc_noise - accuracies_noise[1]:.2f}%")
print(f"  Noise Std=0.10  → Accuracy: {accuracies_noise[2]:.2f}% | Drop: {clean_acc_noise - accuracies_noise[2]:.2f}%")
print(f"  Noise Std=0.20  → Accuracy: {accuracies_noise[4]:.2f}% | Drop: {clean_acc_noise - accuracies_noise[4]:.2f}%")

avg_robustness_noise = np.mean([clean_acc_noise - acc for acc in accuracies_noise[1:4]])
print(f"  Avg Robustness Drop (mild noise): {avg_robustness_noise:.2f}%")

print("\n⚔️  FGSM ADVERSARIAL ATTACK ROBUSTNESS:")
print(f"  Epsilon=0.01   → Accuracy: {accuracies_fgsm[1]:.2f}% | Drop: {clean_acc_fgsm - accuracies_fgsm[1]:.2f}%")
print(f"  Epsilon=0.05   → Accuracy: {accuracies_fgsm[3]:.2f}% | Drop: {clean_acc_fgsm - accuracies_fgsm[3]:.2f}%")
print(f"  Epsilon=0.10   → Accuracy: {accuracies_fgsm[4]:.2f}% | Drop: {clean_acc_fgsm - accuracies_fgsm[4]:.2f}%")

avg_robustness_fgsm = np.mean([clean_acc_fgsm - acc for acc in accuracies_fgsm[1:4]])
print(f"  Avg Robustness Drop (mild attack): {avg_robustness_fgsm:.2f}%")

print("\n💡 INTERPRETATION:")
if avg_robustness_noise < 5:
    print("  ✅ Model is ROBUST to Gaussian noise")
else:
    print(f"  ⚠️  Model shows WEAKNESS to Gaussian noise (avg drop: {avg_robustness_noise:.2f}%)")

if avg_robustness_fgsm < 10:
    print("  ✅ Model is FAIRLY ROBUST to FGSM attacks")
elif avg_robustness_fgsm < 20:
    print(f"  ⚠️  Model is VULNERABLE to FGSM attacks (avg drop: {avg_robustness_fgsm:.2f}%)")
else:
    print(f"  🔴 Model is HIGHLY VULNERABLE to FGSM attacks (avg drop: {avg_robustness_fgsm:.2f}%)")

print("\n" + "="*70)
""");


## Step 9: Perturbation Strength Analysis

In [ ]:
%%jshell

runPy("""
# Visualize perturbation strength
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
fig.suptitle('Understanding Perturbation Strength', fontsize=14, fontweight='bold')

sample_image = sample_images[0:1].cpu().clone()
sample_label = sample_labels[0:1].cpu()

# Row 1: Different noise levels
noise_levels_viz = [0, 0.1, 0.3]
for idx, noise_std in enumerate(noise_levels_viz):
    ax = axes[idx]
    if noise_std == 0:
        img = denormalize(sample_image)[0].permute(1, 2, 0).numpy()
        ax.set_title(f'Clean Image', fontsize=11, fontweight='bold')
    else:
        noisy = add_gaussian_noise(sample_image, noise_std=noise_std)
        img = denormalize(noisy)[0].permute(1, 2, 0).numpy()
        ax.set_title(f'Gaussian Noise (σ={noise_std})', fontsize=11, fontweight='bold')

    ax.imshow(img)
    ax.axis('off')

plt.tight_layout()
plt.savefig('perturbation_strength.png', dpi=150, bbox_inches='tight')
plt.savefig('/content/regression_plot.png')

print("✅ Perturbation strength visualization saved")
""");


## Step 10: Summary & Recommendations

In [ ]:
%%jshell

runPy("""
print("\n" + "="*70)
print("🎯 ADVERSARIAL ROBUSTNESS TESTING REPORT")
print("="*70)

print("\n📌 KEY FINDINGS:")
print(f"\n1. GAUSSIAN NOISE PERFORMANCE:")
print(f"   • Clean Accuracy: {clean_acc_noise:.2f}%")
print(f"   • Accuracy at σ=0.1: {accuracies_noise[2]:.2f}%")
print(f"   • Drop at σ=0.1: {clean_acc_noise - accuracies_noise[2]:.2f}%")

print(f"\n2. FGSM ATTACK PERFORMANCE:")
print(f"   • Clean Accuracy: {clean_acc_fgsm:.2f}%")
print(f"   • Accuracy at ε=0.05: {accuracies_fgsm[3]:.2f}%")
print(f"   • Drop at ε=0.05: {clean_acc_fgsm - accuracies_fgsm[3]:.2f}%")

print(f"\n3. ROBUSTNESS COMPARISON:")
if avg_robustness_noise < avg_robustness_fgsm:
    print(f"   Model is MORE ROBUST to random noise than adversarial attacks")
else:
    print(f"   Model is MORE ROBUST to adversarial attacks than random noise")

print("\n💡 RECOMMENDATIONS TO IMPROVE ROBUSTNESS:")
print("   1. Use adversarial training with FGSM or PGD attacks")
print("   2. Apply data augmentation with noise and small rotations")
print("   3. Increase model capacity slightly for robustness")
print("   4. Use certified defense methods (randomized smoothing)")
print("   5. Test with stronger attacks (PGD, C&W)")

print("\n📚 FOR PRODUCTION DEPLOYMENT:")
if avg_robustness_fgsm > 15:
    print("   🔴 CAUTION: Model is vulnerable to adversarial attacks")
    print("   Recommend adversarial training before deployment")
else:
    print("   ✅ Model shows reasonable robustness but still needs validation")

print("\n" + "="*70)
""");


In [ ]:
%%jshell

runPy("""
import datetime, pytz;
print("Current Time in IST:", datetime.datetime.now(pytz.utc).astimezone(pytz.timezone('Asia/Kolkata')).strftime('%Y-%m-%d %H:%M:%S'))
""");
